# 06 - A tour of SMAClite (StarCraft micromanagement)

**SMAClite** is a lightweight Python re-implementation of the StarCraft Multi-Agent Challenge (SMAC) - the standard benchmark for cooperative MARL -
without needing the StarCraft II game.

* Two armies fight on a small map. **Every unit of our army is one RL agent**; the enemy army is controlled by a simple built-in script
  (it attack-moves towards our starting point).
* Agents see only what is near them, choose from a small set of discrete actions, and the whole team receives **one shared reward**.

This notebook explores the environment: no learning yet, agents act randomly.

In [ ]:
%matplotlib inline
import random
import matplotlib.pyplot as plt
from marl.smac_tools import *
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

## The maps
Each map is a fixed battle scenario. Same armies on both sides (e.g. `2s3z`) = symmetric; otherwise one side is stronger (e.g. `10m_vs_11m`).
`actions` = choices per agent, `obs` = numbers each agent observes, `state` = numbers in the global state used only during training.

In [ ]:
print_table(map_overview())

## Map `2s3z`: 2 Stalkers + 3 Zealots on each side
* **Stalker** - ranged unit (range 6), fast, armoured; deals extra damage to armoured targets.
* **Zealot** - melee unit (has to touch its target), hits twice per attack.
* Both have a **shield** that is damaged before health.

In [ ]:
env = make_smac("2s3z")
print_mdp(env)
print()
print_table(unit_stats(env))

## Time and episodes
* One environment step = **8 game ticks = 0.5 s** of game time. All agents act at the same time.
* The episode ends when one army is dead. SMAClite has **no time limit** - when training we stop episodes after 150 steps.

## Actions (discrete, one per agent per step)
| id | action |
|---|---|
| 0 | no-op - only for dead agents |
| 1 | stop |
| 2-5 | move north / south / east / west (2 distance units) |
| 6 + j | attack enemy *j* (a healer instead heals ally *j*) |

Not every action is allowed at every moment: an **availability mask** removes e.g. attacks on enemies that are dead or out of range (6), or moves into walls.
The agent must pick an available action.

In [ ]:
obs, _ = env.reset(seed=0)
print("at the start, agent 0 can:", available(env, 0))
pick = random_policy(0)
t = 0
while not has_attack(env, 0):             # random moves until an enemy is within agent 0's range
    obs, r, done, _, _ = env.step(pick(env, obs))
    t += 1
print(f"{t} steps later, agent 0 can:", available(env, 0))

## What one agent observes
Each agent sees only units within **sight range 9** (its weapon reaches only 6). All positions are **relative to itself** and divided by the sight range.
Units it cannot see are all zeros. Below: the non-zero features of agent 0's observation right now (`decode(..., nonzero_only=False)` shows all of them).

In [ ]:
print(f"agent 0 observes {env.obs_size} numbers; non-zero right now:")
print_table(decode(obs_feature_names(env, agent=0), obs[0]))

## The global state (only for centralised training)
The state has **every unit on the map** with **absolute** positions (relative to the map centre), health, weapon cooldown, shields, unit types,
plus **the last action of every agent**. Agents never see it when acting; centralised critics/mixers use it during training.

In [ ]:
state = env.get_state()
rows = decode(state_feature_names(env), state)
print(f"state: {env.state_size} numbers; first non-zero ones:")
print_table(rows[:25])

## The reward
**One number per step, given to every agent:**

* damage dealt to enemies (health + shield, after armour)
* **+10** for every enemy killed
* **+200** for winning the battle
* taking damage is **not** punished

The total is scaled so that a perfect episode adds up to **20**. Below, a random episode: the reward split into its three parts,
both teams' health, and how much damage each of our agents actually dealt.

In [ ]:
frames, records = rollout_smac(env, seed=1)
print(f"{len(records)} steps, won: {records[-1]['won']}, return: {sum(r['reward'] for r in records):.2f} / 20")
plot_episode(records, env)
plt.tight_layout()
show_video(frames, fps=6, width=720)

## Map `MMM`: Marines, Marauders and a Medivac
10 agents: 7 **Marines** (cheap, ranged), 2 **Marauders** (ranged, strong against armour) and 1 **Medivac** - a flying healer that **cannot attack**:
its actions are *heal ally j*.

In [ ]:
mmm = make_smac("MMM")
print_mdp(mmm, agent=0)          # agent 0 is the Medivac: its target actions are heals
print()
print_table(unit_stats(mmm))

In [ ]:
frames, records = rollout_smac(mmm, seed=0)
print(f"{len(records)} steps, won: {records[-1]['won']}, return: {sum(r['reward'] for r in records):.2f} / 20")
heal = sum(r['healing'][0] for r in records)
print(f"the Medivac healed {heal:.0f} hp - and the reward contains no term for healing")
plot_episode(records, mmm)
plt.tight_layout()
show_video(frames, fps=6, width=720)